#### Question Answer RNN model 

#### In the project we build a complete RNN network that can answer asked question, even if the question not in data set in same formate 

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('QA_Dataset.csv')

In [3]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [4]:
# tokanization 
def tokenize(text):
    text = text.lower()
    text = text.replace ("?", " ")
    text = text.replace ("'", " ")
    return text.split()

In [5]:
tokenize("What is the capital of Germany?")

['what', 'is', 'the', 'capital', 'of', 'germany']

In [6]:
# vocab
vocab = {'<UNK>': 0}


In [7]:
def build_vocab(row):
   
    tokenized_qestion =tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])

    merze_token = tokenized_qestion + tokenized_answer

    for token in merze_token:
        if token not in vocab:
            vocab[token] = len(vocab)

In [8]:
df.apply(build_vocab, axis= 1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [9]:
# convet word in numaricals 
def txt_to_num (text, vocab):

    index = []

    for token in tokenize(text):
        if token in vocab :
            index.append(vocab[token])
        else:
            index.append(vocab['<UNK>'])

    return index

In [10]:
txt_to_num("ever thing happen for a reason", vocab) 

[0, 0, 0, 39, 14, 0]

In [11]:
import torch
from torch.utils.data import DataLoader, Dataset

In [12]:
class QADataset (Dataset):

    def __init__(self, df,vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        num_question = txt_to_num(self.df.iloc[index]['question'], self.vocab)
        num_answer = txt_to_num(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(num_question), torch.tensor(num_answer)

In [13]:
dataset = QADataset(df, vocab)

In [14]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [15]:
for question, answer in  dataloader:

    print(question, answer)

tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([[276]])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([[85]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([[128]])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([[173]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[10, 96,  3, 97]]) tensor([[98]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([[179]])
tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]]) tensor([[268]])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([[254]])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([[145]])
tensor([[42, 18,  2, 62, 63,  3, 64, 18]]) tensor([[65]])
tensor([[ 10,  96,   3, 104, 239]]) tensor([[240]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([[215]])
tensor([[  1,   2,   3, 

### Architecture of RNN

In [16]:
import torch.nn as nn

In [17]:
class simpleRNN(nn.Module):

    def __init__(self, vocab_size): # constructor 
        super().__init__()
        self.embeding = nn.Embedding(vocab_size, embedding_dim=40)
        self.rnn = nn.RNN(40,64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)


    def forward (self, question):
        embedded_question = self.embeding(question)
        hidden , final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))
        return output

In [18]:
learning_rate = 0.001
epochs = 40

In [19]:
model = simpleRNN(len(vocab))

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [21]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.918405
Epoch: 2, Loss: 455.055875
Epoch: 3, Loss: 378.043538
Epoch: 4, Loss: 323.567418
Epoch: 5, Loss: 276.133056
Epoch: 6, Loss: 230.774832
Epoch: 7, Loss: 188.095804
Epoch: 8, Loss: 150.300680
Epoch: 9, Loss: 117.719177
Epoch: 10, Loss: 91.769395
Epoch: 11, Loss: 71.413875
Epoch: 12, Loss: 56.440266
Epoch: 13, Loss: 45.225491
Epoch: 14, Loss: 36.550395
Epoch: 15, Loss: 30.127519
Epoch: 16, Loss: 25.199911
Epoch: 17, Loss: 21.135419
Epoch: 18, Loss: 18.334606
Epoch: 19, Loss: 15.760554
Epoch: 20, Loss: 13.958267
Epoch: 21, Loss: 12.071703
Epoch: 22, Loss: 10.650062
Epoch: 23, Loss: 9.421166
Epoch: 24, Loss: 8.336095
Epoch: 25, Loss: 7.371630
Epoch: 26, Loss: 6.560648
Epoch: 27, Loss: 5.876239
Epoch: 28, Loss: 5.292089
Epoch: 29, Loss: 4.808872
Epoch: 30, Loss: 4.372215
Epoch: 31, Loss: 3.989138
Epoch: 32, Loss: 3.633788
Epoch: 33, Loss: 3.356211
Epoch: 34, Loss: 3.088140
Epoch: 35, Loss: 2.849055
Epoch: 36, Loss: 2.646984
Epoch: 37, Loss: 2.454934
Epoch: 38, Loss:

In [22]:
def predict (model, question, threshold =0.5) :

    # convert txt to num
    num_question =txt_to_num(question,vocab) 

    # convert to tensors
    question_tensor = torch.tensor(num_question).unsqueeze(0)

    ## send to model
    output = model(question_tensor)

    # convert logits to probability
    prob = torch.nn.functional.softmax(output,dim=1)


    # find the index of max prob
    value, index = torch.max(prob, dim=1)

    if value < threshold:
        print("I don't know")
        return # stops the function here

    print(list(vocab.keys())[index])
    




In [ ]:
predict(model,"What is the capital of france")

paris
